In [ ]:
import requests
from recipe_scrapers import scrape_html
import pandas as pd
import unicodedata
import numpy as np
import re

def convert_to_float(s):
    total = 0.0
    standard_digits = ""
    
    for char in s:
        # 1. Check if the character is a normal digit or a decimal point
        if char.isdigit() or char == '.':
            standard_digits += char
        else:
            try:
                # 2. Check if it's a Unicode fraction (like ¼, ½, etc.)
                # unicodedata.numeric() converts '¼' to 0.25
                total += unicodedata.numeric(char)
            except (TypeError, ValueError):
                # If it's a unit (like 'g' or 'oz'), ignore it or handle it here
                pass
    
    # 3. Add the standard digits to the fraction total
    if standard_digits:
        total += float(standard_digits)
        
    return total

url = input('Provide website to scrape: ')

# 1. Manually get the HTML (recipe-scrapers likes a header to not get blocked)
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.encoding = 'utf-8'
html = response.text

# 2. Use scrape_html with supported_only=False (this is the actual 'wild mode')
scraper = scrape_html(html, org_url=url, supported_only=False)

mappings = {
        "Bread": ["bread", "bagel", "sourdough", "roll", "bun", "focaccia", "dough"],
        "Cookie": ["cookie", "biscuit", "macaron"],
        "Cake": ["cake", "cupcake", "torte"],
        "Pie": ["pie", "tart", "galette"],
        "Pastry": ["muffin", "scone", "croissant", "danish"]
    }

type = []
title = [(scraper.title())]
for key in mappings:
    for identifier in mappings[key]:
        if identifier in scraper.title().lower():
            type.append(key)

ing_combo = (scraper.ingredients())
Ingredients = []
amts = []
Measures = []
for i in ing_combo:
    i_split = i.split(maxsplit=2)
    Ingredients.append(i_split[2])
    amts.append(i_split[0])
    Measures.append(i_split[1])

Amounts = [convert_to_float(a) for a in amts]

Ingredients = [i.title() for i in Ingredients]

Instructions = scraper.instructions_list()
tags = []
# 1. Calculate how many blanks are needed
target_length = len(Ingredients)
if len(Instructions) > target_length:
    target_length = len(Instructions)

ins_padding = target_length - len(Instructions)
name_padding = target_length - len(title)
type_padding = target_length - len(type)
ing_padding = target_length - len(Ingredients)
amount_padding = target_length - len(Amounts)
measure_padding = target_length - len(Measures)
tag_padding = target_length - len(Measures)

# 2. Append the blank strings
# This creates a list like ['', '', '', ...] and adds it to the end
Instructions = Instructions + ([''] * ins_padding)
title = title + ([''] * name_padding)
type = type + ([''] * type_padding)
Ingredients = Ingredients + ([''] * ing_padding)
Amounts = Amounts + ([''] * amount_padding)
Measures = Measures + ([''] * measure_padding)


Amounts = [amt if amt != '' else np.nan for amt in Amounts]
directions = []
for ins in Instructions:
    clean = re.sub(r'[^\x00-\x7f]', r'', ins)
    directions.append(clean)


recipe = {
    'Recipe' : title,
    'Type' : type,
    'Ingredients' : Ingredients,
    'Amount' : Amounts,
    'Measure' : Measures,
    'Instructions' : directions,
    'Tags' : []
}

df = pd.DataFrame(recipe)
df.head()
df.to_csv(fr"C:\Users\Brad Hepburn\Desktop\Recipe Cards\recipes\{title[0]}.csv", index=False)
# df.to_csv(r'C:\Users\Brad Hepburn\Desktop\test.csv', index=False)


ValueError: All arrays must be of the same length